# Visual Text & OCR Extraction Comparison
This notebook compares digital vs scanned text extraction algorithms:
1. **Digital Fallback**: PyMuPDF vs pdfplumber (character bounding boxes & text segments)
2. **Scanned OCR**: EasyOCR vs Tesseract OCR (overlay bounding boxes & text outputs)

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import fitz  # PyMuPDF

pdf_path = 'data/scientific/Scientific_001.pdf'
img_path = 'data/scientific/Scientific_001.png'

if not os.path.exists(pdf_path):
    print("Scientific_001.pdf not found in data/scientific/. Please place it there.")
else:
    doc = fitz.open(pdf_path)
    print(f"Loaded PDF: {pdf_path} | Page Count: {len(doc)}")

In [ ]:
# Comparison 1: Digital Text Fallback
if os.path.exists(pdf_path):
    page = doc[0]
    
    print("--- PyMuPDF Native Text Output (First 200 chars) ---")
    print(page.get_text("text")[:200])
    
    import pdfplumber
    with pdfplumber.open(pdf_path) as pl:
        pl_page = pl.pages[0]
        print("\n--- pdfplumber Native Text Output (First 200 chars) ---")
        print((pl_page.extract_text() or '')[:200])

In [ ]:
# Comparison 2: Scanned OCR on Page Crop
from algorithms.text_extraction.scanned.easyocr.extractor import extract_text as easy_ocr
from algorithms.text_extraction.scanned.tesseract.extractor import extract_text as tesseract_ocr

if os.path.exists(img_path):
    img = Image.open(img_path)
    # Crop a small text region for visual testing (e.g. coordinates [100, 200, 600, 400])
    crop_box = [100, 200, 600, 400]
    cropped = img.crop(crop_box)
    
    print("Running EasyOCR on crop...")
    easy_res = easy_ocr(cropped)
    
    print("Running Tesseract on crop...")
    tess_res = tesseract_ocr(cropped)
    
    print("\nEasyOCR Output:")
    print(easy_res.get('full_text', ''))
    
    print("\nTesseract Output:")
    print(tess_res.get('full_text', ''))
    
    # Show cropped region
    plt.imshow(cropped)
    plt.axis('off')
    plt.show()